In [1]:
import sys
# On installe 'accelerate' et on met à jour 'transformers' pour qu'ils soient synchronisés
!pip install accelerate>=0.26.0 transformers[torch] --upgrade


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from datasets import load_from_disk
from transformers import AutoModelForQuestionAnswering, TrainingArguments, Trainer, DefaultDataCollator, AutoTokenizer

# ==========================================
# 1. CONFIGURATION
# ==========================================
DATA_PATH = "../data/processed/tokenized_data"
OUTPUT_MODEL_DIR = "../src/raq_model" 
MODEL_CHECKPOINT = "bert-base-uncased" 

print("🚀 Démarrage de la phase d'entraînement...")

# ==========================================
# 2. CHARGEMENT DES DONNÉES PRÊTES
# ==========================================
try:
    tokenized_datasets = load_from_disk(DATA_PATH)
    print(f"✅ Dataset chargé : {len(tokenized_datasets['train'])} Train / {len(tokenized_datasets['test'])} Test")
except Exception as e:
    print(f"❌ Erreur chargement données : {e}")
    raise

# ==========================================
# 3. CONFIGURATION DU MODÈLE
# ==========================================
print(f"🏗️ Chargement du modèle : {MODEL_CHECKPOINT}...")
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_CHECKPOINT)

# --- CORRECTION ICI (eval_strategy au lieu de evaluation_strategy) ---
args = TrainingArguments(
    output_dir="checkpoints_temp", 
    eval_strategy="epoch",   # <--- C'EST ICI QUE J'AI CORRIGÉ
    save_strategy="epoch",         
    learning_rate=3e-5,            
    per_device_train_batch_size=4, 
    per_device_eval_batch_size=4,
    num_train_epochs=3,            
    weight_decay=0.01,             
    load_best_model_at_end=True,   
    report_to="none"               
)

data_collator = DefaultDataCollator()

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=None, 
    data_collator=data_collator,
)

# ==========================================
# 4. LANCEMENT
# ==========================================
print("\n🔥 DÉBUT DU FINE-TUNING (Patience...)...")
trainer.train()

print("\n🎉 ENTRAÎNEMENT TERMINÉ !")

# ==========================================
# 5. SAUVEGARDE
# ==========================================
print(f"💾 Sauvegarde dans : {OUTPUT_MODEL_DIR}")
trainer.save_model(OUTPUT_MODEL_DIR)

# On sauvegarde aussi le tokenizer pour l'API
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
tokenizer.save_pretrained(OUTPUT_MODEL_DIR)

print(f"✅ TOUT EST PRÊT ! Le modèle est dans '{OUTPUT_MODEL_DIR}'.")

C:\MASTER\2annee\S3\Deep-Learning\MINI-PROJET\chatbot-educatif-intelligent\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚀 Démarrage de la phase d'entraînement...
✅ Dataset chargé : 136 Train / 35 Test
🏗️ Chargement du modèle : bert-base-uncased...


Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\moufl\AppData\Local\Temp\ipykernel_5720\3075423535.py:46: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(



🔥 DÉBUT DU FINE-TUNING (Patience...)...


C:\MASTER\2annee\S3\Deep-Learning\MINI-PROJET\chatbot-educatif-intelligent\env\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss
1,No log,0.986011
2,No log,0.012660
3,No log,0.004379


C:\MASTER\2annee\S3\Deep-Learning\MINI-PROJET\chatbot-educatif-intelligent\env\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\MASTER\2annee\S3\Deep-Learning\MINI-PROJET\chatbot-educatif-intelligent\env\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



🎉 ENTRAÎNEMENT TERMINÉ !
💾 Sauvegarde dans : ../src/raq_model
✅ TOUT EST PRÊT ! Le modèle est dans '../src/raq_model'.
